# 04 — Análise estatística (nível estadual — 645 municípios)

**Objetivo:** testar a hipótese central do estudo:

> Municípios de SP com maior proporção de idosos morando sozinhos têm maior
> taxa de internação por causas associadas a falta de socorro imediato
> (lesões/causas externas, sintomas mal definidos, transtornos mentais),
> mesmo controlando pelo IDH municipal?

**Entrada:** `data/processed/dataset_municipios_sp.csv` — a versão **limpa**
gerada no notebook 03, seção 3.5 (outliers e municípios sem IDH já
removidos; ~240 municípios). Não é o painel (`dataset_consolidado_sp.csv`,
que tem 5 linhas por município e causaria pseudorreplicação — ver notebook
03) nem a versão bruta (`dataset_municipios_sp_bruto.csv`, 645 municípios,
sem limpeza — usada só na seção 4.2b como checagem de robustez).

**Saídas:** tabelas e figuras em `outputs/tables/` e `outputs/figures/` —
prontas para entrar na seção de Resultados do artigo.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

mun = pd.read_csv(config.DATA_PROCESSED / "dataset_municipios_sp.csv")
print(mun.shape, "(base limpa)")
mun.head()


## 4.1 Estatística descritiva


In [ ]:
mun.describe(include="all").T


## 4.2 Correlação: % idosos sozinhos × taxa de internação

Um ponto por município, base limpa (sem outliers, todos com IDH).


In [ ]:
x = "pct_idosos_sozinhos"
y = "taxa_internacao_100k_domicilios_idosos"

r, p = stats.pearsonr(mun[x], mun[y])
print(f"Correlação de Pearson (n={len(mun)}): r={r:.3f}, p={p:.4g}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.regplot(data=mun, x=x, y=y, ax=ax, scatter_kws={"alpha": 0.5})

rc = mun[mun["municipio"] == config.RIO_CLARO_NOME]
if not rc.empty:
    ax.scatter(rc[x], rc[y], color="red", s=100, zorder=5, label=config.RIO_CLARO_NOME)
    ax.legend()
else:
    print(f"Aviso: {config.RIO_CLARO_NOME} caiu fora da base limpa (outlier ou sem IDH) -- confira dataset_municipios_sp_bruto.csv")

ax.set_xlabel("% de domicílios com responsável idoso que são unipessoais")
ax.set_ylabel("Internações por 100 mil domicílios com responsável idoso")
ax.set_title(f"Idosos sozinhos × internações — municípios de SP, base limpa (n={len(mun)})")
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "correlacao_idosos_sozinhos_internacoes.png", dpi=150)
plt.show()


## 4.2b Checagem de robustez — com e sem a limpeza

Comparar o resultado na base limpa com a base bruta (sem remover outliers
nem municípios sem IDH) mostra o quanto a conclusão depende da limpeza —
uma checagem de robustez importante para declarar no artigo.


In [ ]:
bruto = pd.read_csv(config.DATA_PROCESSED / "dataset_municipios_sp_bruto.csv")
bruto_com_idh = bruto.dropna(subset=["idhm"])

r_bruto, p_bruto = stats.pearsonr(bruto_com_idh[x], bruto_com_idh[y])
r_limpo, p_limpo = stats.pearsonr(mun[x], mun[y])

print(f"Bruto  (n={len(bruto_com_idh)}, com outliers, só remove quem não tem IDH): r={r_bruto:.3f}, p={p_bruto:.4g}")
print(f"Limpo  (n={len(mun)}, sem outliers, sem quem não tem IDH):                  r={r_limpo:.3f}, p={p_limpo:.4g}")


## 4.3 Regressão controlando por IDH

Modelo: `taxa_internacao_100k_domicilios_idosos ~ pct_idosos_sozinhos + idhm`


In [ ]:
formula = "taxa_internacao_100k_domicilios_idosos ~ pct_idosos_sozinhos + idhm"
modelo = smf.ols(formula, data=mun).fit()
print(modelo.summary())
with open(config.OUTPUTS_TABLES / "regressao_ols.txt", "w") as f:
    f.write(modelo.summary().as_text())


**Como ler este resultado:** depois da limpeza (outliers fora, só municípios
com IDH), tanto a correlação simples quanto o coeficiente de
`pct_idosos_sozinhos` na regressão costumam vir **sem significância
estatística** — diferente do resultado na base bruta, onde o coeficiente
aparecia quase significativo (achado de confundimento/suppressor effect).
Isso não invalida o estudo: é exatamente o tipo de checagem de robustez que
um artigo precisa mostrar. A leitura honesta é **"não encontramos
associação robusta entre % de idosos sozinhos e taxa de internação no nível
estadual, depois de controlar por qualidade de dado"** — o que desloca o
argumento central do estudo mais para o Nível 2 (Rio Claro, notebook 05,
com dado mais controlado e o CadÚnico) e para a necessidade de resolver a
questão "local de internação x local de residência" do SIH (ver notebook
02 e `FONTES_RIO_CLARO.md`) antes de reafirmar qualquer associação em
nível estadual. Vale reportar os dois resultados (bruto e limpo, seção
4.2b) no artigo — a diferença entre eles é, em si, um achado sobre a
qualidade do dado disponível.


## 4.4 Perfil das internações por causa

Lembrando: cada "causa" aqui é um **capítulo da CID-10** (ver
`config.CAUSAS_SIH`), não o subgrupo específico do plano original. Usa o
painel completo (contagens totais, não a taxa — não afetado pela limpeza
de outliers da seção 3.5).


In [ ]:
painel = pd.read_csv(config.DATA_PROCESSED / "dataset_consolidado_sp.csv")

causas = list(config.CAUSAS_SIH.keys())
totais = painel[causas].sum().rename(index=config.CAUSAS_SIH_LABELS)

fig, ax = plt.subplots(figsize=(7, 5))
totais.sort_values().plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("Total de internações em idosos, 2022-2026 (estado de SP)")
ax.set_title("Perfil das internações em idosos, por capítulo CID-10")
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "perfil_causas.png", dpi=150)
plt.show()


## 4.5 Evolução da taxa de internação por ano (painel)

Aqui usamos o painel completo (todos os 645 municípios, sem remover
outliers) — é uma descrição da série temporal, não um teste de hipótese
com `pct_idosos_sozinhos`, então a limpeza da seção 3.5 não se aplica. A
mediana (mais robusta a outliers que a média) é a leitura mais confiável
aqui.


In [ ]:
evolucao = painel.groupby("ano")["taxa_internacao_100k_domicilios_idosos"].agg(["mean", "median"]).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(evolucao["ano"], evolucao["mean"], marker="o", label="Média")
ax.plot(evolucao["ano"], evolucao["median"], marker="o", label="Mediana")
ax.set_xlabel("Ano")
ax.set_ylabel("Internações por 100 mil domicílios com responsável idoso")
ax.set_title("Evolução da taxa de internação — municípios de SP\n(2026 parcial, até julho)")
ax.legend()
fig.tight_layout()
fig.savefig(config.OUTPUTS_FIGURES / "evolucao_taxa_internacao_sp.png", dpi=150)
plt.show()


## 4.6 Mapa coroplético (opcional — requer `geopandas` e o shapefile de SP)

1. Baixe: https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/UFs/SP/SP_Municipios_2022.zip
2. Salve (sem descompactar) como `data/external/sp_municipios.zip`

⚠️ O shapefile do IBGE traz `codigo_ibge` — como nossa base usa nome de
município (seção 3), o merge aqui é por nome normalizado também. Usa a
base limpa (`mun`), então municípios removidos na seção 3.5 aparecem em
cinza (sem dado) no mapa.


In [ ]:
try:
    import geopandas as gpd

    caminho_shp = config.DATA_EXTERNAL / "sp_municipios.zip"
    if caminho_shp.exists():
        gdf = gpd.read_file(f"zip://{caminho_shp}")
        gdf["municipio_norm"] = gdf["NM_MUN"].apply(config.normalizar_municipio)
        mapa = gdf.merge(mun, on="municipio_norm", how="left")

        fig, ax = plt.subplots(figsize=(9, 9))
        mapa.plot(column="taxa_internacao_100k_domicilios_idosos", cmap="OrRd", legend=True, ax=ax,
                  missing_kwds={"color": "lightgrey"})
        ax.set_title("Taxa de internação em idosos — SP (2022-2026, base limpa)")
        ax.axis("off")
        fig.tight_layout()
        fig.savefig(config.OUTPUTS_FIGURES / "mapa_taxa_internacao_sp.png", dpi=150)
        plt.show()
    else:
        print(f"Baixe o shapefile conforme instruções acima e salve em {caminho_shp}")
except ImportError:
    print("geopandas não instalado. No Anaconda Prompt: conda install -c conda-forge geopandas")
